In [15]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

In [16]:
# 1. Load Data
# Dataset Anda menggunakan delimiter ';'
df = pd.read_csv('student_data.csv', sep=';')

In [17]:
# 2. Preprocessing
# Pisahkan Fitur (X) dan Target (y)
X = df.drop('Target', axis=1).values
y_raw = df['Target'].values

In [18]:
# Encode Target (Dropout, Enrolled, Graduate -> 0, 1, 2)
le = LabelEncoder()
y = le.fit_transform(y_raw)
num_classes = len(np.unique(y))

# Scaling (Sangat PENTING untuk Neural Network)
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [19]:
# 3. Split Data (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Hitung Class Weights untuk menangani ketidakseimbangan data
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y), y=y)
class_weights = torch.tensor(class_weights, dtype=torch.float32)

# 5. Convert ke PyTorch Tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [20]:
# 6. Buat DataLoader
batch_size = 32

train_dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = torch.utils.data.TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Data Loaded. Features: {X.shape[1]}, Classes: {num_classes}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

Data Loaded. Features: 36, Classes: 3
Device: cpu


In [21]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score, accuracy_score

# --- Definisi Model LSTM ---
class TabularLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=2, dropout=0.3):
        super(TabularLSTM, self).__init__()
        
        # LSTM layer
        # Kita proyeksikan input tabular menjadi sequence virtual
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # Batch normalization untuk stabilitas
        self.bn_input = nn.BatchNorm1d(input_dim)
        
        # LSTM: input_size=input_dim (karena kita treat 1 step time series)
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, 
                            batch_first=True, dropout=dropout, bidirectional=True)
        
        # Fully Connected Layer akhir
        self.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, 64), # *2 karena Bidirectional
            nn.ReLU(),
            nn.Linear(64, output_dim)
        )

    def forward(self, x):
        x = self.bn_input(x)
        # Reshape input menjadi (Batch, Sequence_Length, Features)
        # Disini kita anggap Sequence Length = 1 untuk tabular sederhana
        x = x.unsqueeze(1) 
        
        # LSTM output
        lstm_out, (hn, cn) = self.lstm(x)
        
        # Ambil output dari step terakhir
        out = lstm_out[:, -1, :]
        out = self.fc(out)
        return out

# --- Setup Training ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
input_dim = X_train.shape[1]
output_dim = num_classes

model_lstm = TabularLSTM(input_dim, hidden_dim=128, output_dim=output_dim).to(device)

# Loss dengan Class Weight (Handling Imbalance)
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = optim.AdamW(model_lstm.parameters(), lr=0.001, weight_decay=1e-4)

# --- Training Loop dengan Early Stopping ---
best_f1 = 0
patience = 10
trigger_times = 0
epochs = 100

print("Mulai Training LSTM...")

for epoch in range(epochs):
    model_lstm.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model_lstm(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    # Evaluasi setiap epoch
    model_lstm.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model_lstm(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    
    print(f"Epoch {epoch+1}/{epochs} | Loss: {running_loss/len(train_loader):.4f} | Val Acc: {acc:.4f} | Val F1: {f1:.4f}")
    
    # Early Stopping Check
    if f1 > best_f1:
        best_f1 = f1
        trigger_times = 0
        # Simpan model terbaik jika mau: torch.save(model_lstm.state_dict(), 'best_lstm.pth')
    else:
        trigger_times += 1
        if trigger_times >= patience:
            print(f"Early stopping di epoch {epoch+1}")
            break

print(f"Training Selesai. Best F1 Score LSTM: {best_f1:.4f}")

Mulai Training LSTM...
Epoch 1/100 | Loss: 0.8587 | Val Acc: 0.7175 | Val F1: 0.7320
Epoch 2/100 | Loss: 0.7042 | Val Acc: 0.7514 | Val F1: 0.7572
Epoch 3/100 | Loss: 0.6840 | Val Acc: 0.7458 | Val F1: 0.7528
Epoch 4/100 | Loss: 0.6769 | Val Acc: 0.7040 | Val F1: 0.7233
Epoch 5/100 | Loss: 0.6676 | Val Acc: 0.7141 | Val F1: 0.7289
Epoch 6/100 | Loss: 0.6628 | Val Acc: 0.7096 | Val F1: 0.7255
Epoch 7/100 | Loss: 0.6546 | Val Acc: 0.7277 | Val F1: 0.7406
Epoch 8/100 | Loss: 0.6367 | Val Acc: 0.7130 | Val F1: 0.7305
Epoch 9/100 | Loss: 0.6349 | Val Acc: 0.7209 | Val F1: 0.7352
Epoch 10/100 | Loss: 0.6239 | Val Acc: 0.7254 | Val F1: 0.7406
Epoch 11/100 | Loss: 0.6086 | Val Acc: 0.7311 | Val F1: 0.7432
Epoch 12/100 | Loss: 0.6052 | Val Acc: 0.7198 | Val F1: 0.7335
Early stopping di epoch 12
Training Selesai. Best F1 Score LSTM: 0.7572
